# 04 - Ringkasan Hasil

Tabel 3, plot komparatif F1, analisis segmentation failures, feature importance.

In [ ]:
# ============================================================
# Setup cell - Kaggle Notebooks (Kaggle-only). Jalankan PALING ATAS.
# Cara attach dataset: panel kanan > + Add Data > cari
#   'fruit and vegetable disease healthy vs rotten' > Add.
# ============================================================
import os
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import sys
import shutil
import subprocess
from pathlib import Path

# 1. Clone repo dari GitHub (atau pull jika sudah ada di sesi ini)
REPO_URL = "https://github.com/faizhuda/pcd-kelompok-17.git"
PROJECT_DIR = Path("/kaggle/working/pcd-kelompok-17")
if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=False)

# 2. Working directory ke root project + tambah ke sys.path
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# 3. Dependency inti SUDAH pre-installed di Kaggle -> tidak ada pip install.

# 4. Dataset gambar (read-only, hasil + Add Data)
# Auto-detect: Kaggle bisa mount di /kaggle/input/<slug> atau
# /kaggle/input/datasets/<user>/<slug> tergantung cara attach.
_DATASET_SLUG = 'fruit-and-vegetable-disease-healthy-vs-rotten'
_candidates = [
    Path('/kaggle/input') / _DATASET_SLUG,
    Path('/kaggle/input/datasets/muhammad0subhan') / _DATASET_SLUG,
]
RAW_DATA_DIR = next((p for p in _candidates if p.exists()), None)
if RAW_DATA_DIR is None:
    # Fallback: cari folder mana saja di /kaggle/input yang berisi gambar dataset
    for _p in Path('/kaggle/input').rglob(_DATASET_SLUG):
        if _p.is_dir():
            RAW_DATA_DIR = _p
            break
assert RAW_DATA_DIR is not None, "Dataset belum di-attach. + Add Data > cari dataset > Add."

# 5. Auto-restore hasil notebook sebelumnya (untuk notebook 03 & 04).
#    Attach output run lama via: + Add Data > Your Work / Dataset bersama.
def restore_previous_outputs():
    # Kaggle mounts notebook outputs di /kaggle/input/notebooks/<user>/<notebook>/
    # sehingga perlu rglob, bukan glob satu level.
    restored = []
    for repo in Path("/kaggle/input").rglob("pcd-kelompok-17"):
        if not repo.is_dir():
            continue
        for sub in ("results", "data/processed"):
            src_dir = repo / sub
            if src_dir.exists():
                shutil.copytree(src_dir, PROJECT_DIR / sub, dirs_exist_ok=True)
                restored.append(str(src_dir))
    return restored

restored = restore_previous_outputs()
print("Project :", PROJECT_DIR)
print("Dataset :", RAW_DATA_DIR)
print("Restore :", restored or "(mulai dari nol)")


In [ ]:
import os
import sys
from pathlib import Path

# Setup cell sudah chdir ke PROJECT_DIR & menambah sys.path (Kaggle-only).
ROOT = Path("/kaggle/working/pcd-kelompok-17")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.evaluate import aggregate_feature_importance, plot_feature_importance, print_summary_table
from src.utils import get_project_paths, read_best_enhancement

paths = get_project_paths()
metrics_dir = paths["metrics"]


## Tabel 3 - Ringkasan Semua Skenario

In [ ]:
summary = print_summary_table(metrics_dir)
summary


## Plot Komparatif F1-Score

In [ ]:
if not summary.empty:
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(data=summary, x="scenario_id", y="f1_weighted", hue="model", ax=ax)
    ax.set_title("F1-Score (weighted) per Skenario")
    ax.set_xlabel("Skenario")
    plt.tight_layout()
    plt.savefig(paths["figures"] / "f1_comparison.png", dpi=150)
    plt.show()


## Enhancement Terpilih (E*)

In [ ]:
print(f"Best enhancement: {read_best_enhancement(metrics_dir)}")


## Segmentation Failures

In [ ]:
fail_path = metrics_dir / "segmentation_failures.csv"
if fail_path.exists():
    fails = pd.read_csv(fail_path)
    display(fails.groupby("commodity").size().sort_values(ascending=False).head(10))
else:
    print("Belum ada log segmentasi. Jalankan skenario dengan segmentasi aktif.")


## Uji Signifikansi McNemar

In [ ]:
sig_path = metrics_dir / "significance_tests.csv"
if sig_path.exists():
    display(pd.read_csv(sig_path))
else:
    print("Jalankan notebook 02 dan 03 untuk uji McNemar.")


## Feature Importance (Skenario 9 - RF)

In [ ]:
import joblib

rf_path = paths["models"] / "rf_scenario_09.pkl"
if rf_path.exists():
    rf = joblib.load(rf_path)
    from src.features import get_feature_group_names
    names = get_feature_group_names("all", segmented=True)
    if len(names) != len(rf.feature_importances_):
        names = [f"f{i}" for i in range(len(rf.feature_importances_))]
    labels, vals = aggregate_feature_importance(rf.feature_importances_, names)
    plot_feature_importance(vals, labels, save_path=paths["figures"] / "feature_importance_s09.png")
    plt.show()
else:
    print("Model RF S9 belum tersedia. Jalankan notebook 02 terlebih dahulu.")


## Inference Time Comparison

In [ ]:
if not summary.empty and "inference_time_ms_per_image" in summary.columns:
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(data=summary, x="scenario_id", y="inference_time_ms_per_image", ax=ax)
    ax.set_title("Inference Time (ms/image)")
    plt.tight_layout()
    plt.show()


## Per-Commodity Performance Comparison

In [ ]:
s5_pred_path = metrics_dir / "predictions_s5.csv"
s10_pred_path = metrics_dir / "predictions_s10.csv"

if s5_pred_path.exists() and s10_pred_path.exists():
    from sklearn.metrics import f1_score
    s5_preds = pd.read_csv(s5_pred_path)
    s10_preds = pd.read_csv(s10_pred_path)
    label_map = {"fresh": 0, "rotten": 1}
    s5_preds["true_encoded"] = s5_preds["label"].map(label_map)
    s10_preds["true_encoded"] = s10_preds["label"].map(label_map)
    s5_comm = []
    for comm, group in s5_preds.groupby("commodity"):
        f1 = f1_score(group["true_encoded"], group["pred"], average="weighted", zero_division=0)
        s5_comm.append({"commodity": comm, "samples": len(group), "f1_s5": f1})
    df_s5_comm = pd.DataFrame(s5_comm)
    s10_comm = []
    for comm, group in s10_preds.groupby("commodity"):
        f1 = f1_score(group["true_encoded"], group["pred"], average="weighted", zero_division=0)
        s10_comm.append({"commodity": comm, "f1_s10": f1})
    df_s10_comm = pd.DataFrame(s10_comm)
    df_compare = pd.merge(df_s5_comm, df_s10_comm, on="commodity").sort_values("f1_s10", ascending=False)
    display(df_compare)
    df_melted = df_compare.melt(id_vars=["commodity", "samples"], value_vars=["f1_s5", "f1_s10"],
                                var_name="model", value_name="f1_score")
    df_melted["model"] = df_melted["model"].map({"f1_s5": "S5 SVM", "f1_s10": "S10 CNN"})
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(data=df_melted, x="commodity", y="f1_score", hue="model", ax=ax)
    ax.set_title("F1-Score per Komoditas (S5 SVM vs S10 CNN)")
    ax.set_ylabel("Weighted F1-Score")
    ax.set_xlabel("Komoditas")
    ax.set_ylim(0, 1.05)
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(paths["figures"] / "commodity_comparison.png", dpi=150)
    plt.show()
else:
    print("Prediksi S5 atau S10 belum lengkap. Lewati perbandingan komoditas.")
